# Legal KB Experiment — Full Pipeline (single notebook)

Run end-to-end trên Google Colab Pro (T4 GPU):
1. **Setup** (~30 min): GDrive, deps, Neo4j Aura, push KG, smoke test
2. **Inference** (~2-3h): 5 R1 arms + 6 R2 combos parallel
3. **Metrics + Report** (~2h): compute metrics, audit fixes, generate reports
4. **Visualizations**: charts, heatmaps, distributions

**Total wall time: ~5-6h** (vs ~13h local laptop).

**Run All** sau khi config secrets — không cần babysit.

## Prerequisites

1. Push code repo lên GDrive: `/MyDrive/legal-graph-kb`
2. Neo4j Aura free instance (https://aura.io)
3. Colab secrets:
   - `OPENAI_API_KEY`
   - `NEO4J_URI` (neo4j+s://xxx.databases.neo4j.io)
   - `NEO4J_USER` (neo4j)
   - `NEO4J_PASSWORD`

## ⚙️ CONFIG — adjust trước khi Run All

**All knobs exposed**: R1 generator, R2 models, judge model, scope flags.

In [ ]:
# ═══════════════════ MODEL CONFIG (đầy đủ knobs) ═══════════════════

# R1 — generator model cho 5 arms (graphrag/llm_only/elite_no_retrieval/ontology/graphrag)
# Mặc định gpt-4o-mini (cheap). Có thể thử: gpt-4.1, gpt-4o, gpt-5-mini
R1_MODEL       = 'gpt-4o-mini'

# R2 — models để test trong multi-model experiment (2 elite arms × N models)
# Add/remove models tự do. OpenAI models: gpt-4.1, gpt-4o, gpt-5-mini, gpt-5, o3-mini
R2_MODELS      = ['gpt-4.1', 'gpt-4o', 'gpt-5-mini']

# JUDGE — model dùng cho LLM-as-Judge metrics (faithfulness, halluc, pairwise, ...)
# Khuyến nghị gpt-4o-mini (rẻ + consistent). Cho rigorous: gpt-4o full.
JUDGE_MODEL    = 'gpt-4o-mini'

# ═══════════════════ SCOPE CONFIG ═══════════════════
N_QUESTIONS    = 200    # max 200 (set 10 cho pilot)
RUN_R1         = True   # chạy 5-arm experiment
RUN_R2         = True   # chạy multimodel experiment
BACKFILL_PLAIN = True   # backfill plain_answer cho elite (~$0.72)
MAX_PARALLEL   = 5      # số combos chạy concurrent (limit theo OpenAI tier RPM)

# ═══════════════════ PATHS ═══════════════════
REPO_GDRIVE    = '/content/drive/MyDrive/legal-graph-kb'
RESULTS_GDRIVE = '/content/drive/MyDrive/legal-graph-kb-results'

print(f'R1 generator: {R1_MODEL}')
print(f'R2 models:    {R2_MODELS}')
print(f'Judge model:  {JUDGE_MODEL}')
print(f'N questions:  {N_QUESTIONS}')
print(f'Total combos: {(5 if RUN_R1 else 0) + (2*len(R2_MODELS) if RUN_R2 else 0)}')

## 📊 Supported Metrics — Reference Table

Cell này document tất cả metrics pipeline tính (no execution).

| # | Metric | Category | Paper ref | Áp dụng |
|---|---|---|---|---|
| 1 | `citation_validity` | Deterministic | — | All arms — % cit_id tồn tại trong KG |
| 2 | `citation_recall` | Deterministic | Liu EMNLP'23 | All — % sentence có ≥1 cit |
| 3 | `citation_precision` | LLM-judge | Liu EMNLP'23 | All — % cit thực sự support claim |
| 4 | `faithfulness` | LLM-judge (RAGAS) | Es EACL'24 | All — % claim được support bởi cited text |
| 5 | `answer_relevance` | RAGAS + embed | Es EACL'24 | All (elite: trên plain_answer) |
| 6 | `content_hallucination_rate` | LLM-judge | Magesh JELS'25 | All — % claim misstate+unsup / n_claims |
| 7 | `invented_citation_rate` | Deterministic | Magesh JELS'25 | All — % cit_id KHÔNG có trong KG |
| 8 | `hallucination_rate` (legacy) | LLM-judge | Magesh JELS'25 | All — combined formula, kept for backward compat |
| 9 | `bertscore_f1` (p, r) | Embedding | Zhang ICLR'20 | All (elite: trên plain_answer khi có) |
| 10 | `cost_usd` | Deterministic | — | All — tính từ prompt + completion tokens × model pricing |
| 11 | `latency_s` | Deterministic | — | All — wall time per inference |
| 12 | `prolog_success_rate` | Deterministic | Pan EMNLP'23 | Elite only — % Prolog query returns solution |
| 13 | `first_try_success_rate` | Deterministic | Pan EMNLP'23 | Elite only — % success với 0 repair |
| 14 | `repair_invoked_rate` | Deterministic | Pan EMNLP'23 | Elite only — % cần ≥1 repair |
| 15 | `avg_repair_rounds` | Deterministic | Pan EMNLP'23 | Elite only — mean rounds (max=2) |
| 16 | `pairwise consensus` | LLM-judge | Zheng NeurIPS'23 | Non-baseline arms vs baseline (position swap) |
| 17 | **API error rate** | Deterministic | Audit 2026-05-26 | All — tag records có prompt_tokens==0 (silent connection error) |

**Statistical tests** (compute_significance.py):
- **McNemar** cho paired binary outcomes (Prolog success, pairwise wins)
- **Bootstrap 95% CI** (10k resamples) cho continuous metrics (faithfulness diff)
- **Bonferroni correction** tại α = 0.05 / n_claims (mặc định 5 claims → α_bonf = 0.01)

**Aggregate modes**:
- **Macro**: mean of per-record rates (default)
- **Micro**: corpus-level Σ correct / Σ extracted (cho citation metrics)
- **Insufficient flag**: cells với n_valid < 30 → 'insufficient' (sample size too small)

# Phase 1 — Setup

## 1.1 GPU + GDrive + secrets

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv

from google.colab import drive, userdata
import os, sys, subprocess, time, json
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
drive.mount('/content/drive')

for k in ['OPENAI_API_KEY', 'NEO4J_URI', 'NEO4J_USER', 'NEO4J_PASSWORD']:
    os.environ[k] = userdata.get(k) or ''
os.environ['NEO4J_DATABASE'] = 'neo4j'
os.environ['EMBED_DEVICE'] = 'cuda'
os.environ['PYTHONIOENCODING'] = 'utf-8'
# OPENAI_MODEL được set per-phase (inference → R1_MODEL, judge → JUDGE_MODEL)
print({k: ('set' if os.environ.get(k) else 'MISSING') for k in
       ['OPENAI_API_KEY', 'NEO4J_URI', 'NEO4J_USER', 'NEO4J_PASSWORD']})

## 1.2 Copy repo + install deps + persistent results symlink

In [ ]:
REPO_DIR = '/content/legal-graph-kb'
if not os.path.exists(REPO_DIR):
    !cp -r $REPO_GDRIVE $REPO_DIR
%cd $REPO_DIR
sys.path.insert(0, REPO_DIR)

# Install deps (Colab có sẵn pandas/matplotlib/numpy)
!pip install -q neo4j sentence-transformers openai python-dotenv bert-score tqdm 2>&1 | tail -3
!apt-get install -y -q swi-prolog 2>&1 | tail -2

# Persistent results dir → GDrive (survive session reboot)
os.makedirs(f'{RESULTS_GDRIVE}/data/eval', exist_ok=True)
if os.path.islink('data/eval') or not os.path.exists('data/eval'):
    !rm -rf data/eval && ln -s $RESULTS_GDRIVE/data/eval data/eval
!ls -la data/eval | head -3

## 1.3 Neo4j Aura — verify + push KG nếu chưa có

In [ ]:
from neo4j import GraphDatabase
driver = GraphDatabase.driver(os.environ['NEO4J_URI'],
                               auth=(os.environ['NEO4J_USER'], os.environ['NEO4J_PASSWORD']))
with driver.session(database='neo4j') as s:
    n_art = s.run('MATCH (n:Article) RETURN count(n) AS c').single()['c']
    n_cla = s.run('MATCH (n:Clause) RETURN count(n) AS c').single()['c']
    print(f'Aura: {n_art} Articles, {n_cla} Clauses')
    has_vec = any('clause_vec' in (i.get('name') or '') for i in s.run('SHOW INDEXES').data())
    if not has_vec:
        s.run('''CREATE VECTOR INDEX clause_vec IF NOT EXISTS
                 FOR (c:Clause) ON c.embedding
                 OPTIONS {indexConfig: {`vector.dimensions`: 1024,
                                         `vector.similarity_function`: 'cosine'}}''')
        print('Created clause_vec index')
    else:
        print('clause_vec index exists ✓')
driver.close()

if n_art == 0:
    print('Pushing KG to Aura...')
    !python -m src.load_neo4j 2>&1 | tail -10

## 1.4 Smoke test với R1_MODEL

In [ ]:
os.environ['OPENAI_MODEL'] = R1_MODEL  # set cho R1 inference
from src.rag_query import RagPipeline
from experiments.elite_pipelines import EliteGraphRAGPipeline

Q = 'Tôi đã đóng BHXH 12 năm, có đủ điều kiện hưởng lương hưu không?'

p = RagPipeline(); _ = p.embed_model
r = p.ask(Q, top_k=8, verbose=False)
print(f'[graphrag × {R1_MODEL}] answer[:200]: {r.answer[:200]}')
print(f'  citations: {r.citation_ids} | elapsed: {r.elapsed_s}s\n')
p.close()

ep = EliteGraphRAGPipeline(model=R1_MODEL)
ans = ep.ask(Q)
print(f'[elite_graphrag × {R1_MODEL}] IRAC[:200]: {ans.answer[:200]}')
print(f'  plain_answer: {ans.plain_answer}')
ep.close()

# Phase 2 — Inference (parallel)

Each subprocess inherits OPENAI_MODEL env. R1 dùng R1_MODEL, R2 dùng explicit `--models` flag.

In [ ]:
os.makedirs('logs', exist_ok=True)

def run_combo(cmd, name, env_overrides=None):
    """Run shell cmd với optional env overrides per-job."""
    t0 = time.time()
    env = os.environ.copy()
    if env_overrides:
        env.update(env_overrides)
    with open(f'logs/{name}.log', 'w', encoding='utf-8') as f:
        proc = subprocess.run(cmd, shell=True, stdout=f,
                              stderr=subprocess.STDOUT, env=env)
    return name, proc.returncode, time.time() - t0

def parallel(cmds_with_env, max_workers):
    """cmds_with_env = list of (cmd, name, env_dict)."""
    t0 = time.time()
    with ThreadPoolExecutor(max_workers=max_workers) as ex:
        futs = {ex.submit(run_combo, c, n, e): n for c, n, e in cmds_with_env}
        for fut in as_completed(futs):
            n, code, dur = fut.result()
            status = '✓' if code == 0 else f'✗ exit={code}'
            print(f'[{time.strftime("%H:%M:%S")}] {status} {n} ({dur:.0f}s)')
    print(f'\nWall time: {(time.time()-t0)/60:.1f} min')

# Build cmd list — R1 jobs share R1_MODEL via OPENAI_MODEL env
cmds = []
if RUN_R1:
    for arm in ['graphrag', 'llm_only', 'elite_no_retrieval', 'elite_ontology', 'elite_graphrag']:
        cmds.append((
            f'python -m experiments.run_inference --arm {arm} --n {N_QUESTIONS}',
            f'r1_{arm}',
            {'OPENAI_MODEL': R1_MODEL},
        ))

# R2 jobs — explicit --models flag (override env via flag)
if RUN_R2:
    for arm in ['elite_no_retrieval', 'elite_graphrag']:
        for model in R2_MODELS:
            cmds.append((
                f'python -m experiments.run_multimodel_inference '
                f'--arms {arm} --models {model} --n {N_QUESTIONS}',
                f'r2_{arm}_{model.replace(".","_")}',
                None,
            ))

print(f'Launching {len(cmds)} combos parallel (max {MAX_PARALLEL} concurrent)...\n')
parallel(cmds, max_workers=MAX_PARALLEL)

## 2.x Progress check

In [ ]:
for root in ['data/eval/results', 'data/eval/multimodel/results']:
    if not Path(root).exists(): continue
    print(f'\n{root}:')
    for d in sorted(Path(root).iterdir()):
        if d.is_dir():
            n = len([f for f in d.glob('A*.json') if not f.name.endswith('.error.json')])
            n_api = sum(1 for f in d.glob('A*.json')
                        if not f.name.endswith('.error.json')
                        and json.loads(f.read_text(encoding='utf-8')).get('prompt_tokens') == 0)
            api_tag = f' ({n_api} api_err)' if n_api else ''
            print(f'  {d.name:<40} {n}/{N_QUESTIONS}{api_tag}')

# Phase 3 — Metrics + Report

Switch OPENAI_MODEL → JUDGE_MODEL cho metric compute (compute_metrics đọc env này).

## 3.1 (Optional) Backfill plain_answer cho elite arms

In [ ]:
if BACKFILL_PLAIN:
    # Backfill dùng gpt-4o-mini (rẻ, đủ cho rephrasing IRAC → prose)
    os.environ['BACKFILL_MODEL'] = 'gpt-4o-mini'
    !python -m experiments.rerender_plain_answer --combos all 2>&1 | tail -15
else:
    print('Skipped (BACKFILL_PLAIN=False)')

## 3.2 Compute metrics (R1 + R2 parallel) — judge = JUDGE_MODEL

In [ ]:
metric_cmds = []
judge_env = {'OPENAI_MODEL': JUDGE_MODEL}  # compute_metrics đọc env này cho judge
if RUN_R1:
    metric_cmds.append(('python -m experiments.compute_metrics', 'metrics_r1', judge_env))
if RUN_R2:
    metric_cmds.append(('python -m experiments.compute_multimodel_metrics', 'metrics_r2', judge_env))
parallel(metric_cmds, max_workers=2)

## 3.3 Audit fixes (v1: pairwise + v2: halluc split + api_err tag)

In [ ]:
!python -m experiments.audit_apply_fixes 2>&1 | tail -10
!python -m experiments.audit_apply_fixes_v2 2>&1 | tail -10

## 3.4 Generate reports + significance tests

In [ ]:
!python -m experiments.generate_report 2>&1 | tail -3
!python -m experiments.generate_multimodel_report 2>&1 | tail -3
!python -m experiments.compute_significance 2>&1 | tail -3

# Phase 4 — Visualizations 📈

Charts + heatmaps + distributions từ `metrics.json` data thật (no fabrication).

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

plt.rcParams['figure.dpi'] = 100
sns.set_style('whitegrid')

def ch(rec, *keys):
    v = rec
    for k in keys:
        if not isinstance(v, dict): return None
        v = v.get(k)
    return v

def flatten_metrics(metrics_path, source_label):
    """Load metrics.json → long-format DataFrame."""
    data = json.loads(Path(metrics_path).read_text(encoding='utf-8'))
    rows = []
    for arm, recs in data.items():
        for r in recs:
            if r.get('api_error'): continue  # exclude API errors
            rows.append({
                'source': source_label,
                'arm': r.get('arm', arm),
                'model': r.get('model', ''),
                'combo': arm,
                'stt': r.get('stt'),
                'citation_validity': ch(r, 'citation_validity', 'validity_rate'),
                'citation_recall': ch(r, 'citation_recall', 'recall'),
                'citation_precision': ch(r, 'citation_precision', 'precision'),
                'faithfulness': ch(r, 'faithfulness', 'faithfulness'),
                'content_hallu': ch(r, 'hallucination', 'content_hallucination_rate'),
                'invented_cit': ch(r, 'hallucination', 'invented_citation_rate'),
                'answer_relevance': ch(r, 'answer_relevance', 'answer_relevance'),
                'bertscore_f1': ch(r, 'bertscore', 'bertscore_f1'),
                'cost_usd': ch(r, 'cost', 'cost_usd'),
                'latency_s': ch(r, 'latency', 'latency_s'),
                'prolog_success': 1 if ch(r, 'prolog_rollback', 'prolog_success') else 0
                                   if ch(r, 'prolog_rollback', 'prolog_success') is not None else None,
                'first_try_success': 1 if ch(r, 'prolog_rollback', 'first_try_success') else 0
                                      if ch(r, 'prolog_rollback', 'first_try_success') is not None else None,
            })
    return pd.DataFrame(rows)

df_r1 = flatten_metrics('data/eval/metrics.json', 'R1') if os.path.exists('data/eval/metrics.json') else pd.DataFrame()
df_r2 = flatten_metrics('data/eval/multimodel/metrics.json', 'R2') if os.path.exists('data/eval/multimodel/metrics.json') else pd.DataFrame()
df_all = pd.concat([df_r1, df_r2], ignore_index=True)
print(f'R1: {len(df_r1)} records, R2: {len(df_r2)} records, Total clean: {len(df_all)}')
df_all.head()

## 4.1 Bar chart — Prolog success per combo (Elite arms only)

In [ ]:
elite = df_all[df_all['arm'].str.startswith('elite_') & df_all['prolog_success'].notna()].copy()
if not elite.empty:
    agg = elite.groupby('combo').agg(
        prolog_success_rate=('prolog_success', 'mean'),
        first_try_rate=('first_try_success', 'mean'),
        n=('prolog_success', 'count'),
    ).reset_index().sort_values('prolog_success_rate', ascending=False)
    fig, ax = plt.subplots(figsize=(11, 5))
    x = np.arange(len(agg))
    ax.bar(x - 0.2, agg['prolog_success_rate'], 0.4, label='success_rate', color='#2ca02c')
    ax.bar(x + 0.2, agg['first_try_rate'], 0.4, label='first_try_rate', color='#1f77b4')
    ax.set_xticks(x); ax.set_xticklabels(agg['combo'], rotation=30, ha='right')
    ax.set_ylim(0, 1.05); ax.set_ylabel('rate')
    ax.set_title('Prolog Reliability — success vs first-try (API errors excluded)')
    ax.legend()
    for i, row in agg.iterrows():
        ax.text(i, row['prolog_success_rate']+0.01,
                f"n={row['n']}", ha='center', fontsize=8, color='gray')
    plt.tight_layout(); plt.show()
else:
    print('No elite records')

## 4.2 Heatmap — Mean metric × combo

In [ ]:
METRIC_COLS = ['citation_validity', 'citation_recall', 'citation_precision',
                'faithfulness', 'content_hallu', 'invented_cit',
                'bertscore_f1']
if not df_all.empty:
    heat = df_all.groupby('combo')[METRIC_COLS].mean()
    fig, ax = plt.subplots(figsize=(11, max(4, 0.4*len(heat))))
    sns.heatmap(heat, annot=True, fmt='.3f', cmap='RdYlGn',
                vmin=0, vmax=1, cbar_kws={'label': 'mean value'}, ax=ax)
    ax.set_title('Metric Matrix — mean per combo (clean records)')
    ax.set_xlabel(''); ax.set_ylabel('')
    plt.tight_layout(); plt.show()

## 4.3 Scatter — Cost vs Faithfulness (per combo)

In [ ]:
if not df_all.empty:
    agg = df_all.groupby('combo').agg(
        cost_usd=('cost_usd', 'mean'),
        faithfulness=('faithfulness', 'mean'),
        latency_s=('latency_s', 'mean'),
        n=('stt', 'count'),
    ).reset_index().dropna()
    fig, ax = plt.subplots(figsize=(10, 6))
    scatter = ax.scatter(agg['cost_usd'], agg['faithfulness'],
                          s=agg['latency_s']*20, alpha=0.6,
                          c=range(len(agg)), cmap='tab20')
    for _, row in agg.iterrows():
        ax.annotate(row['combo'], (row['cost_usd'], row['faithfulness']),
                     fontsize=8, xytext=(5,5), textcoords='offset points')
    ax.set_xlabel('Mean cost (USD per question)')
    ax.set_ylabel('Mean faithfulness')
    ax.set_title('Cost vs Faithfulness (bubble size = latency)')
    ax.set_xscale('log')
    plt.tight_layout(); plt.show()

## 4.4 Box plot — Latency distribution per combo

In [ ]:
if not df_all.empty:
    fig, ax = plt.subplots(figsize=(12, 5))
    df_plot = df_all.dropna(subset=['latency_s']).copy()
    order = df_plot.groupby('combo')['latency_s'].median().sort_values().index.tolist()
    sns.boxplot(data=df_plot, x='combo', y='latency_s', order=order, ax=ax)
    ax.set_yscale('log')
    ax.set_title('Latency distribution per combo (log scale)')
    ax.set_xlabel(''); ax.set_ylabel('seconds')
    plt.xticks(rotation=30, ha='right')
    plt.tight_layout(); plt.show()

## 4.5 Stacked bar — Pairwise consensus distribution

In [ ]:
# Pairwise: R1 vs graphrag baseline + R2 GR vs NR per model
rows = []
if os.path.exists('data/eval/metrics.json'):
    r1 = json.loads(Path('data/eval/metrics.json').read_text(encoding='utf-8'))
    for arm, recs in r1.items():
        if arm == 'graphrag': continue
        for r in recs:
            if r.get('api_error'): continue
            pw = r.get('pairwise_vs_baseline')
            if pw:
                rows.append({'compare': f'{arm} vs graphrag', 'consensus': pw['consensus']})

if os.path.exists('data/eval/multimodel/metrics.json'):
    r2 = json.loads(Path('data/eval/multimodel/metrics.json').read_text(encoding='utf-8'))
    for combo, recs in r2.items():
        if not combo.startswith('elite_graphrag__'): continue
        model = combo.replace('elite_graphrag__', '').replace('_', '.')
        for r in recs:
            if r.get('api_error'): continue
            pw = r.get('pairwise_vs_no_retrieval')
            if pw:
                rows.append({'compare': f'GR vs NR ({model})', 'consensus': pw['consensus']})

if rows:
    df_pw = pd.DataFrame(rows)
    pivot = df_pw.groupby(['compare', 'consensus']).size().unstack(fill_value=0)
    pivot_pct = pivot.div(pivot.sum(axis=1), axis=0) * 100
    fig, ax = plt.subplots(figsize=(11, 5))
    pivot_pct.plot(kind='barh', stacked=True, ax=ax, colormap='tab20')
    ax.set_xlabel('% of records')
    ax.set_title('Pairwise consensus distribution (clean records)')
    ax.legend(loc='center left', bbox_to_anchor=(1.0, 0.5), fontsize=8)
    plt.tight_layout(); plt.show()

## 4.6 Sortable DataFrame — full metric matrix

In [ ]:
if not df_all.empty:
    summary = df_all.groupby('combo').agg(
        n=('stt', 'count'),
        cit_validity=('citation_validity', 'mean'),
        cit_recall=('citation_recall', 'mean'),
        cit_precision=('citation_precision', 'mean'),
        faithfulness=('faithfulness', 'mean'),
        content_hallu=('content_hallu', 'mean'),
        invented_cit=('invented_cit', 'mean'),
        bertscore=('bertscore_f1', 'mean'),
        cost_usd=('cost_usd', 'mean'),
        latency_s=('latency_s', 'mean'),
        prolog_success=('prolog_success', 'mean'),
    ).round(4).sort_index()
    # Style nhẹ
    summary.style.background_gradient(cmap='RdYlGn', axis=0,
                                       subset=['cit_validity','cit_recall','cit_precision',
                                               'faithfulness','bertscore','prolog_success'])\
                  .background_gradient(cmap='RdYlGn_r', axis=0,
                                       subset=['content_hallu','invented_cit','cost_usd','latency_s'])

## 4.7 Display markdown reports inline

In [ ]:
from IPython.display import Markdown, display
for f in ['reports/FINAL_REPORT.md', 'reports/significance.md']:
    if os.path.exists(f):
        print(f'\n{"="*70}\n{f}\n{"="*70}')
        display(Markdown(open(f, encoding='utf-8').read()[:6000]))

## 4.8 Sync reports → GDrive + download zip

In [ ]:
!mkdir -p $RESULTS_GDRIVE/reports
!rsync -av reports/ $RESULTS_GDRIVE/reports/ 2>&1 | tail -5
!cd /content/legal-graph-kb && zip -rq /content/reports_bundle.zip reports/ data/eval/metrics.json data/eval/multimodel/metrics.json data/eval/metrics.csv

from google.colab import files
files.download('/content/reports_bundle.zip')
print('\n✓ Pipeline + visualizations complete')